In [ ]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [ ]:
df = pd.read_csv("../dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = format_context_with_boundaries(df, lang="be")

In [ ]:
df

In [ ]:
with open("../../pronunciation_counts/pronunciation_counts_be.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="be").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [ ]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

In [ ]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

In [ ]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [ ]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [ ]:
pd.DataFrame(data).to_csv("surprisal_beus.csv", index=False)